# Model comparison: CNN vs Faster R-CNN vs YOLO
Task: name the piece (41 elements; shape + LEGO color derived from it).
- **CNN** (ResNet18) classifies a crop around the *true* box → upper bound for a detector + classifier pipeline.
- **Faster R-CNN** and **YOLO11** find the piece *and* name it from the full photo.

Same split for all: inside every photo run, frames in order → first 70% train, next 15% val, last 15% test (consecutive frames are near-duplicates; a random split would leak).

Note: these are single-piece photos. All three will likely score high here; reading built towers (overlap, occlusion) is the real test and needs structure photos.

**What this notebook does.** Compares three ways of naming the piece in a photo, all trained and tested on exactly the same split: a ResNet18 classifier on the brick crop, YOLO11s, and Faster R-CNN. It reports element, shape and color accuracy, box quality and speed.

**Needs:** `data/boxes.csv`, `data/ranges.csv`, `data/element_map.csv`, `dataset/`. **Writes:** `cache1024/`, `crops224/`, `yolo41/`, `models/cnn_resnet18.pt`, `models/frcnn_r50.pt`, `runs/.../cmp_yolo/`.

Setup and the train/val/test split. Inside every photo session (a run in `data/ranges.csv`), the first 70% of frames are train, the next 15% val and the last 15% test, so near-duplicate frames never end up on both sides.

In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')   # all paths below are from the repo root
import csv, os, glob, time, random, shutil
from collections import Counter, defaultdict
from multiprocessing import Pool
import cv2, numpy as np, torch, torchvision
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEV = 'cuda'
W0, H0 = 4032, 3024
S = 1024 / W0                      # cached images are 1024x768
os.makedirs('models', exist_ok=True)

boxes = {r['path']: (r['element_id'], *[float(r[k]) for k in ('x0', 'y0', 'x1', 'y1')]) for r in csv.DictReader(open('data/boxes.csv'))}
ranges = [(int(r['start']), int(r['end'])) for r in csv.DictReader(open('data/ranges.csv'))]
em = {r['element_id']: r for r in csv.DictReader(open('data/element_map.csv'))}
CLASSES = sorted({b[0] for b in boxes.values()}); CI = {c: i for i, c in enumerate(CLASSES)}

def run_of(p):
    n = int(os.path.basename(p)[4:8])
    return next(i for i, (a, b) in enumerate(ranges) if a <= n <= b)

split, runs = {}, defaultdict(list)
for p in boxes: runs[run_of(p)].append(p)
for ps in runs.values():
    ps.sort(); n = len(ps)
    for i, p in enumerate(ps):
        split[p] = 'train' if i < 0.7 * n else 'val' if i < 0.85 * n else 'test'
P = {k: sorted(p for p in boxes if split[p] == k) for k in ('train', 'val', 'test')}
print(len(CLASSES), 'classes |', {k: len(v) for k, v in P.items()})
assert all(Counter(boxes[p][0] for p in P['test'])[c] > 0 for c in CLASSES), 'every class must be in test'

41 classes | {'train': 4564, 'val': 977, 'test': 950}


**Result:** 41 classes, split into 4564 train, 977 val and 950 test photos. Every class appears in the test set.

## Cache: 1024x768 copies (decoding 12 MP JPEGs every epoch is the bottleneck)

In [2]:
CACHE = 'cache1024'
def resize(p):
    out = f'{CACHE}/{p}'
    if not os.path.exists(out):
        os.makedirs(os.path.dirname(out), exist_ok=True)
        im = cv2.imread(p, cv2.IMREAD_REDUCED_COLOR_2)   # 2016x1512, decodes ~4x faster
        cv2.imwrite(out, cv2.resize(im, (1024, 768), interpolation=cv2.INTER_AREA), [cv2.IMWRITE_JPEG_QUALITY, 95])
    return out

CROPS = 'crops224'
def crop(p):
    out = f'{CROPS}/{p}'
    if not os.path.exists(out):
        os.makedirs(os.path.dirname(out), exist_ok=True)
        im = cv2.imread(f'{CACHE}/{p}')
        _, x0, y0, x1, y1 = [v * S if i else v for i, v in enumerate(boxes[p])]
        s = int(max(x1 - x0, y1 - y0) * 1.15) | 1   # square crop + 15% margin keeps the shape's aspect
        c = cv2.getRectSubPix(im, (s, s), ((x0 + x1) / 2, (y0 + y1) / 2))
        cv2.imwrite(out, cv2.resize(c, (224, 224), interpolation=cv2.INTER_AREA))
    return out

with Pool() as pool:
    cached = dict(zip(boxes, pool.map(resize, list(boxes), chunksize=32)))
    crops = dict(zip(boxes, pool.map(crop, list(boxes), chunksize=32)))
print(len(cached), 'cached,', len(crops), 'crops')

6491 cached, 6491 crops


**Result:** All 6491 photos now have a 1024x768 copy and a 224x224 brick crop on disk, so training no longer decodes 12 MP JPEGs.

## Shared evaluation
For every test photo a model returns `(class index or None, box in original pixels or None, number of boxes)`.

In [3]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

results = {}
def evaluate(name, preds, secs):
    true = [CI[boxes[p][0]] for p in P['test']]
    cls = [q[0] for q in preds]
    elem = np.mean([c == t for c, t in zip(cls, true)])
    shape = np.mean([c is not None and em[CLASSES[c]]['part_num'] == em[CLASSES[t]]['part_num'] for c, t in zip(cls, true)])
    color = np.mean([c is not None and em[CLASSES[c]]['color'] == em[CLASSES[t]]['color'] for c, t in zip(cls, true)])
    r = dict(element=elem, shape=shape, color=color, ms_per_img=1000 * secs / len(P['test']))
    if preds[0][1] is not None or any(q[1] is not None for q in preds):   # detectors
        ious = [iou(q[1], boxes[p][1:]) if q[1] is not None else 0 for q, p in zip(preds, P['test'])]
        r.update(box_iou=np.mean(ious), box_iou_over_0_5=np.mean(np.array(ious) > 0.5), exactly_one_box=np.mean([q[2] == 1 for q in preds]))
    results[name] = r
    print(name, {k: round(v, 3) for k, v in r.items()})
    conf = Counter((CLASSES[t], CLASSES[c] if c is not None else 'none') for c, t in zip(cls, true) if c != t)
    for (t, c), n in conf.most_common(8):
        print(f'   {n:3d}x  {t} {em[t]["part_name"][:28]:28s} {em[t]["color"]:18s} -> {c} ' + (f'{em[c]["part_name"][:28]} {em[c]["color"]}' if c in em else ''))

**Result:** `evaluate()` is now defined. Every model below is scored with it in the same way.

## 1. CNN (ResNet18 on crops)
No hue/saturation augmentation: color is part of the class.

In [4]:
import torchvision.transforms.v2 as T

class Crops(Dataset):
    def __init__(s, ps, tf): s.ps, s.tf = ps, tf
    def __len__(s): return len(s.ps)
    def __getitem__(s, i):
        p = s.ps[i]
        im = torch.from_numpy(cv2.imread(crops[p])[..., ::-1].copy()).permute(2, 0, 1)
        return s.tf(im), CI[boxes[p][0]]

norm = T.Compose([T.ToDtype(torch.float32, scale=True), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
train_tf = T.Compose([T.RandomRotation(180), T.RandomResizedCrop(224, scale=(0.7, 1), ratio=(0.9, 1.1)),
                      T.RandomHorizontalFlip(), T.ColorJitter(brightness=0.3, contrast=0.3), norm])
loader = lambda k, tf, sh: DataLoader(Crops(P[k], tf), batch_size=64, shuffle=sh, num_workers=8)

def cnn_predict(model, ps):
    model.eval(); out = []
    with torch.no_grad(), torch.autocast('cuda'):
        for x, _ in DataLoader(Crops(ps, norm), batch_size=128, num_workers=8):
            out += model(x.to(DEV)).argmax(1).tolist()
    return out

CNN_PT = 'models/cnn_resnet18.pt'
cnn = torchvision.models.resnet18(weights='DEFAULT'); cnn.fc = torch.nn.Linear(512, len(CLASSES)); cnn = cnn.to(DEV)
if not os.path.exists(CNN_PT):
    EP = 15; tl = loader('train', train_tf, True)
    opt = torch.optim.AdamW(cnn.parameters(), 1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=EP * len(tl))
    scaler = torch.amp.GradScaler(); best_acc = 0
    for ep in range(EP):
        cnn.train()
        for x, y in tl:
            with torch.autocast('cuda'):
                loss = torch.nn.functional.cross_entropy(cnn(x.to(DEV)), y.to(DEV))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        acc = np.mean(np.array(cnn_predict(cnn, P['val'])) == [CI[boxes[p][0]] for p in P['val']])
        print(f'epoch {ep+1}/{EP} loss {loss.item():.3f} val acc {acc:.3f}')
        if acc >= best_acc: best_acc = acc; torch.save(cnn.state_dict(), CNN_PT)
cnn.load_state_dict(torch.load(CNN_PT))
t = time.time(); c = cnn_predict(cnn, P['test']); secs = time.time() - t
evaluate('CNN (true crop)', [(k, None, None) for k in c], secs)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/denkata/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 72.0MB/s]


epoch 1/15 loss 0.717 val acc 0.881
epoch 2/15 loss 0.151 val acc 0.963
epoch 3/15 loss 0.055 val acc 0.887
epoch 4/15 loss 0.240 val acc 0.775
epoch 5/15 loss 0.126 val acc 0.951
epoch 6/15 loss 0.006 val acc 0.981
epoch 7/15 loss 0.014 val acc 0.988
epoch 8/15 loss 0.004 val acc 0.982
epoch 9/15 loss 0.009 val acc 0.999
epoch 10/15 loss 0.022 val acc 1.000
epoch 11/15 loss 0.018 val acc 0.999
epoch 12/15 loss 0.007 val acc 1.000
epoch 13/15 loss 0.005 val acc 1.000
epoch 14/15 loss 0.001 val acc 1.000
epoch 15/15 loss 0.002 val acc 1.000
CNN (true crop) {'element': np.float64(0.991), 'shape': np.float64(0.991), 'color': np.float64(1.0), 'ms_per_img': 1.053}
     9x  4211199 Brick Sloped 45° 2 x 1 with  Reddish Brown      -> 4211197 Brick Sloped 33° 3 x 1 Reddish Brown


**Result:** The CNN reaches about 100% validation accuracy after ~10 epochs. It is given the true box, so it shows the best a detect-then-classify pipeline could do.

## 2. YOLO11s (41 classes, full photo)

In [5]:
from ultralytics import YOLO
Y = 'yolo41'
shutil.rmtree(Y, ignore_errors=True)
for p in boxes:
    k, stem = split[p], p.replace('/', '_')[:-4]
    os.makedirs(f'{Y}/{k}/images', exist_ok=True); os.makedirs(f'{Y}/{k}/labels', exist_ok=True)
    os.link(cached[p], f'{Y}/{k}/images/{stem}.jpg')
    e, x0, y0, x1, y1 = boxes[p]
    open(f'{Y}/{k}/labels/{stem}.txt', 'w').write(f'{CI[e]} {(x0+x1)/2/W0:.6f} {(y0+y1)/2/H0:.6f} {(x1-x0)/W0:.6f} {(y1-y0)/H0:.6f}\n')
open(f'{Y}/data.yaml', 'w').write(f'path: {os.path.abspath(Y)}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n'
                                   + ''.join(f'  {i}: "{c}"\n' for i, c in enumerate(CLASSES)))

found = glob.glob('runs/**/cmp_yolo/weights/best.pt', recursive=True)   # ultralytics may nest under runs/detect/
if not found:   # ~30-40 min
    YOLO('yolo11s.pt').train(data=f'{Y}/data.yaml', imgsz=1024, epochs=40, batch=8, cache='ram', hsv_h=0, hsv_s=0.2,
                             degrees=180, project=os.path.abspath('runs'), name='cmp_yolo', exist_ok=True, verbose=False)
    found = glob.glob('runs/**/cmp_yolo/weights/best.pt', recursive=True)
yolo = YOLO(found[0])
m = yolo.val(data=f'{Y}/data.yaml', split='test', imgsz=1024, verbose=False)
print(f'YOLO test mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}')

preds = []; t = time.time()
for i in range(0, len(P['test']), 16):
    for res in yolo.predict([cached[p] for p in P['test'][i:i+16]], imgsz=1024, conf=0.25, verbose=False):
        b = res.boxes.cpu()
        if len(b):
            k = int(b.conf.argmax()); preds.append((int(b.cls[k]), (b.xyxy[k] / S).tolist(), len(b)))
        else: preds.append((None, None, 0))
evaluate('YOLO11s', preds, time.time() - t)

Ultralytics 8.4.215 🚀 Python-3.12.16 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 3060, 11907MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo41/data.yaml, degrees=180, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0, hsv_s=0.2, hsv_v=0.4, imgsz=1024, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cmp_yolo, nbs=64, nms=None, opset=None, optimize=False, optimize

**Result:** YOLO is trained (or loaded) and scored on the test photos. Its results line is printed at the end of the cell.

## 3. Faster R-CNN (ResNet50-FPN, COCO-pretrained)

In [6]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

class Det(Dataset):
    def __init__(s, ps, train): s.ps, s.train = ps, train
    def __len__(s): return len(s.ps)
    def __getitem__(s, i):
        p = s.ps[i]; im = cv2.imread(cached[p])[..., ::-1].copy()
        e, *b = boxes[p]; b = np.array([b]) * S
        if s.train and random.random() < 0.5:
            im = im[:, ::-1].copy(); b[:, [0, 2]] = 1024 - b[:, [2, 0]]
        return (torch.from_numpy(im).permute(2, 0, 1).float() / 255,
                {'boxes': torch.tensor(b, dtype=torch.float32), 'labels': torch.tensor([CI[e] + 1])})   # 0 = background

coll = lambda batch: tuple(zip(*batch))
FR_PT = 'models/frcnn_r50.pt'
frcnn = fasterrcnn_resnet50_fpn(weights='DEFAULT', min_size=768, max_size=1024)
frcnn.roi_heads.box_predictor = FastRCNNPredictor(frcnn.roi_heads.box_predictor.cls_score.in_features, len(CLASSES) + 1)
frcnn = frcnn.to(DEV)

def frcnn_predict(ps):
    frcnn.eval(); out = []
    with torch.no_grad():
        for x, _ in DataLoader(Det(ps, False), batch_size=8, num_workers=8, collate_fn=coll):
            for o in frcnn([im.to(DEV) for im in x]):
                keep = o['scores'] > 0.5
                if keep.any():
                    k = int(o['scores'].argmax())   # sorted by score
                    out.append((int(o['labels'][k]) - 1, (o['boxes'][k].cpu() / S).tolist(), int(keep.sum())))
                else: out.append((None, None, 0))
    return out

if not os.path.exists(FR_PT):   # ~30-40 min
    EP = 10; tl = DataLoader(Det(P['train'], True), batch_size=4, shuffle=True, num_workers=8, collate_fn=coll)
    opt = torch.optim.SGD([q for q in frcnn.parameters() if q.requires_grad], lr=0.01, momentum=0.9, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 0.01, total_steps=EP * len(tl))
    scaler = torch.amp.GradScaler(); best_acc = 0
    for ep in range(EP):
        frcnn.train()
        for x, y in tl:
            with torch.autocast('cuda'):
                loss = sum(frcnn([im.to(DEV) for im in x], [{k: v.to(DEV) for k, v in t.items()} for t in y]).values())
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        acc = np.mean([q[0] == CI[boxes[p][0]] for q, p in zip(frcnn_predict(P['val']), P['val'])])
        print(f'epoch {ep+1}/{EP} loss {loss.item():.3f} val acc {acc:.3f}')
        if acc >= best_acc: best_acc = acc; torch.save(frcnn.state_dict(), FR_PT)
frcnn.load_state_dict(torch.load(FR_PT))
t = time.time(); preds = frcnn_predict(P['test']); evaluate('Faster R-CNN', preds, time.time() - t)

/tmp/ipykernel_26421/2193225628.py:43: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


epoch 1/10 loss 0.164 val acc 0.901
epoch 2/10 loss 0.064 val acc 1.000
epoch 3/10 loss 0.059 val acc 1.000
epoch 4/10 loss 0.067 val acc 1.000
epoch 5/10 loss 0.029 val acc 1.000
epoch 6/10 loss 0.049 val acc 1.000
epoch 7/10 loss 0.054 val acc 1.000
epoch 8/10 loss 0.038 val acc 1.000
epoch 9/10 loss 0.033 val acc 1.000
epoch 10/10 loss 0.011 val acc 1.000
Faster R-CNN {'element': np.float64(0.991), 'shape': np.float64(0.991), 'color': np.float64(1.0), 'ms_per_img': 69.521, 'box_iou': np.float64(0.935), 'box_iou_over_0_5': np.float64(1.0), 'exactly_one_box': np.float64(1.0)}
     9x  4211199 Brick Sloped 45° 2 x 1 with  Reddish Brown      -> 4211197 Brick Sloped 33° 3 x 1 Reddish Brown


**Result:** Faster R-CNN reaches 100% validation accuracy from epoch 2. It is the most accurate at placing boxes, but also the slowest.

## Comparison

Print all results in one table.

In [ ]:
keys = ['element', 'shape', 'color', 'box_iou', 'box_iou_over_0_5', 'exactly_one_box', 'ms_per_img']
print(f'{"model":18s}' + ''.join(f'{k:>18s}' for k in keys))
for n, r in results.items():
    print(f'{n:18s}' + ''.join(f'{r[k]:18.3f}' if k in r else f'{"-":>18s}' for k in keys))

model                        element             shape             color           box_iou  box_iou_over_0_5   exactly_one_box        ms_per_img
CNN (true crop)                0.991             0.991             1.000                 -                 -                 -             1.053
YOLO11s                        0.991             0.991             1.000             0.892             0.998             1.000            15.562
Faster R-CNN                   0.991             0.991             1.000             0.935             1.000             1.000            69.521


: 

**Result:** All three models name 99.1% of the test pieces correctly, and get the color right 100% of the time. They differ only in boxes and speed: Faster R-CNN has the best boxes (IoU 0.94) but takes 70 ms per image, YOLO 16 ms, and the CNN 1 ms (without a detector).

## Conclusion
On single-brick photos the task is basically solved: all three models score 99.1% on the element and 100% on color, so accuracy can't decide between them. Speed and box quality can: YOLO is about 4x faster than Faster R-CNN with boxes that are nearly as good. As the opening note warns, this test is too easy. The real question is towers, where bricks are small, touching and partly hidden. That is what `test_real.ipynb` and the notebooks after it explore.